# 🍷 Predicción de Precios de Vino Español — EDA y Modelado

**Proyecto:** Spanish Wine Quality (Kaggle: `fedesoriano/spanish-wine-quality-dataset`)  
**Autor/a:** Tania  
**Objetivo:** predecir el precio (€) de un vino a partir de su puntuación, nº de reseñas, cuerpo, acidez, región y tipo.

**Contenido**
1. Carga y preparación de datos
2. Análisis exploratorio (EDA)
3. Preparación para el modelado
4. Modelos: Regresión Lineal → Random Forest → RF regularizado → RF optimizado (GridSearchCV)
5. Comparativa y selección del modelo final
6. Importancia de variables
7. Análisis de residuos
8. Guardado y verificación del modelo
9. Experimento opcional: transformación logarítmica del precio
10. Conclusiones

## 1. Carga y preparación de datos

In [ ]:
from pathlib import Path

import joblib
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import plot_tree

# Configuración general
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.float_format", "{:.3f}".format)

RANDOM_STATE = 42
MODELS_DIR = Path("models")          # carpeta relativa a este notebook (funciona en cualquier equipo)
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
# Descarga del dataset desde Kaggle y carga con pandas
path = kagglehub.dataset_download("fedesoriano/spanish-wine-quality-dataset")
print("Ruta de descarga de Kaggle:", path)

df = pd.read_csv(Path(path) / "wines_SPA.csv")
print("Dimensiones del DataFrame:", df.shape)
print("Columnas originales:", df.columns.tolist())
display(df.head())

In [ ]:
# Traducción de columnas al español
column_rename_mapping = {
    "wine": "nombre_vino",
    "winery": "bodega",
    "year": "anio",
    "rating": "puntuacion",
    "num_reviews": "num_reseñas",
    "price": "precio",
    "body": "cuerpo",
    "acidity": "acidez",
    "country": "pais",
    "region": "region",
    "type": "tipo",
}
df_esp = df.rename(columns=column_rename_mapping)

print("--- Columnas traducidas ---")
print(df_esp.columns.tolist())
display(df_esp.head(1).T)

### Limpieza de datos
Se eliminan las filas con nulos en las variables clave (target y predictoras numéricas).
Se reporta cuántas filas se pierden, porque es un dato relevante sobre la calidad del dataset.

In [ ]:
cols_clave = ["precio", "puntuacion", "num_reseñas", "cuerpo", "acidez"]

print("Nulos por columna clave (antes de limpiar):")
print(df_esp[cols_clave].isna().sum())

filas_antes = len(df_esp)
df_esp = df_esp.dropna(subset=cols_clave).copy()
filas_perdidas = filas_antes - len(df_esp)

print(f"\nFilas antes: {filas_antes} | después: {len(df_esp)} "
      f"| eliminadas: {filas_perdidas} ({filas_perdidas / filas_antes:.1%})")

print("\n--- Información general ---")
df_esp.info()

print("\n--- Resumen estadístico ---")
display(df_esp.describe())

## 2. Análisis exploratorio (EDA)

Puntos a observar:
- **Precio:** distribución muy asimétrica (mediana baja, cola larga de vinos de colección).
- **Puntuación:** muy poca variabilidad (casi todos los vinos tienen 4.2), lo que limita su poder predictivo.

In [ ]:
# Visualización 1: distribución del precio (escala original y logarítmica)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df_esp["precio"], kde=True, color="purple", bins=50, ax=axes[0])
axes[0].set_title("Distribución del precio (escala original)")
axes[0].set_xlabel("Precio (€)")
axes[0].set_ylabel("Cantidad de vinos")

sns.histplot(np.log1p(df_esp["precio"]), kde=True, color="teal", bins=50, ax=axes[1])
axes[1].set_title("Distribución de log(1 + precio)")
axes[1].set_xlabel("log(1 + precio)")
axes[1].set_ylabel("Cantidad de vinos")

plt.tight_layout()
plt.show()

print(f"Mediana: {df_esp['precio'].median():.2f} € | Media: {df_esp['precio'].mean():.2f} € "
      f"| Máximo: {df_esp['precio'].max():.2f} €")

In [ ]:
# Visualización 2: relación puntuación vs precio
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df_esp, x="puntuacion", y="precio", alpha=0.5, color="b")
plt.title("Relación entre la puntuación y el precio del vino")
plt.xlabel("Puntuación (rating)")
plt.ylabel("Precio (€)")
plt.show()

In [ ]:
# Visualización 3: matriz de correlación de variables numéricas
plt.figure(figsize=(8, 5))
sns.heatmap(df_esp.select_dtypes(include=[np.number]).corr(),
            annot=True, cmap="Blues", fmt=".2f", linewidths=0.5)
plt.title("Matriz de correlación de variables numéricas")
plt.show()

In [ ]:
# Visualización 4: top 10 regiones por precio medio
top_regiones = df_esp.groupby("region")["precio"].mean().nlargest(10).reset_index()

plt.figure(figsize=(8, 4))
sns.barplot(data=top_regiones, x="precio", y="region", hue="region", palette="viridis", legend=False)
plt.title("Top 10 regiones por precio promedio")
plt.xlabel("Precio promedio (€)")
plt.ylabel("Región")
plt.show()

## 3. Preparación para el modelado

- **Target:** `precio`
- **Predictoras:** `puntuacion`, `num_reseñas`, `cuerpo`, `acidez` + `region` y `tipo` (codificadas con *one-hot*).
- Partición 80 % entrenamiento / 20 % prueba con semilla fija para reproducibilidad.

> Nota: `anio` no se incluye como predictora (además tiene nulos y tipo texto).

In [ ]:
features_to_use = ["puntuacion", "num_reseñas", "cuerpo", "acidez", "region", "tipo"]

df_model = df_esp[features_to_use + ["precio"]].copy()
df_model = pd.get_dummies(df_model, columns=["region", "tipo"], drop_first=True)

X = df_model.drop(columns=["precio"])
y = df_model["precio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Nº de columnas tras one-hot: {X.shape[1]}")

## 4. Modelos

Se define una única función de evaluación que calcula **R², RMSE y MAE en Train y Test**,
la brecha relativa de R² (indicador de sobreajuste) y la variación de RMSE y MAE.
Así todos los modelos se miden igual y no hay código duplicado.

**Regla de lectura:** brecha de R² < 5 % → buena generalización.

In [ ]:
def evaluar_modelo(modelo, nombre):
    """Calcula métricas en Train y Test y la brecha de sobreajuste."""
    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)

    r2_train = r2_score(y_train, pred_train)
    r2_test = r2_score(y_test, pred_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_test = mean_absolute_error(y_test, pred_test)

    return {
        "Modelo": nombre,
        "R2_Train": r2_train,
        "R2_Test": r2_test,
        "Brecha_R2_%": (r2_train - r2_test) / r2_train * 100,
        "RMSE_Train": rmse_train,
        "RMSE_Test": rmse_test,
        "Var_RMSE_%": (rmse_test - rmse_train) / rmse_train * 100,
        "MAE_Train": mae_train,
        "MAE_Test": mae_test,
        "Var_MAE_%": (mae_test - mae_train) / mae_train * 100,
    }


def mostrar(res):
    """Imprime un resumen legible de los resultados de un modelo."""
    print(f"--- {res['Modelo']} ---")
    print(f"R²   Train: {res['R2_Train']:.4f} | Test: {res['R2_Test']:.4f} | Brecha: {res['Brecha_R2_%']:.2f}%")
    print(f"RMSE Train: {res['RMSE_Train']:.2f} € | Test: {res['RMSE_Test']:.2f} € | Var: {res['Var_RMSE_%']:+.2f}%")
    print(f"MAE  Train: {res['MAE_Train']:.2f} € | Test: {res['MAE_Test']:.2f} € | Var: {res['Var_MAE_%']:+.2f}%")


resultados = []   # aquí se acumulan los resultados de cada modelo

### 📈 Modelo 1: Regresión Lineal (línea base)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

res_lr = evaluar_modelo(lr_model, "1. Regresión Lineal (línea base)")
resultados.append(res_lr)
mostrar(res_lr)

### 🌳 Modelo 2: Random Forest base (sin regularizar)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)

res_rf = evaluar_modelo(rf_model, "2. Random Forest base")
resultados.append(res_rf)
mostrar(res_rf)

**Diagnóstico:** una brecha de R² tan marcada entre Train y Test indica que el Random Forest
**memorizó ruido** del conjunto de entrenamiento en lugar de aprender patrones generales (sobreajuste severo).

Para reducirla se actúa sobre:
- **Hiperparámetros:** menor `max_depth`, mayor `min_samples_split` y `min_samples_leaf` (suavizan las predicciones).
- **Aleatoriedad entre árboles:** `max_features` menor que el total.
- **Validación cruzada** para elegir la configuración de forma robusta.

### 🌳 Modelo 2.1: Random Forest regularizado

In [ ]:
rf_regularizado = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,            # limita la profundidad de los árboles
    min_samples_split=10,    # más muestras para dividir un nodo
    min_samples_leaf=4,      # más muestras por hoja final
    max_features="sqrt",     # subconjunto aleatorio de variables en cada división
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_regularizado.fit(X_train, y_train)

res_rf_reg = evaluar_modelo(rf_regularizado, "2.1 Random Forest regularizado")
resultados.append(res_rf_reg)
mostrar(res_rf_reg)

### 🔎 Modelo 3: Random Forest optimizado con GridSearchCV

Búsqueda de hiperparámetros con validación cruzada de 5 folds sobre el conjunto de entrenamiento
(16 combinaciones × 5 folds = 80 ajustes), maximizando R².

In [ ]:
param_grid = {
    "n_estimators": [150],
    "max_depth": [8, 10],
    "min_samples_split": [15, 30],
    "min_samples_leaf": [10, 15],
    "max_features": ["sqrt", 0.5],
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1,
)

print("Iniciando la búsqueda de hiperparámetros...")
grid_search.fit(X_train, y_train)

print("\nMejores hiperparámetros:", grid_search.best_params_)
print(f"Mejor R² medio en validación cruzada (CV=5): {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_

res_grid = evaluar_modelo(best_rf, "3. Random Forest GridSearchCV")
resultados.append(res_grid)
print()
mostrar(res_grid)

## 5. Comparativa y selección del modelo final

Se comparan todos los modelos con las mismas métricas. El criterio de selección es:
**mayor R² en Test con brecha < 5 %**, revisando también el MAE.

In [ ]:
df_comparativo = pd.DataFrame(resultados).set_index("Modelo")
display(df_comparativo)

# Gráfico R² Train vs Test
ax = df_comparativo[["R2_Train", "R2_Test"]].plot(kind="bar", figsize=(10, 5), color=["#6a5acd", "#2e8b57"])
ax.set_title("R² en Train vs Test por modelo")
ax.set_ylabel("R²")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Registro de modelos entrenados (para recuperar el seleccionado por nombre)
modelos = {
    res_lr["Modelo"]: lr_model,
    res_rf["Modelo"]: rf_model,
    res_rf_reg["Modelo"]: rf_regularizado,
    res_grid["Modelo"]: best_rf,
}

# Selección automática: mejor R² Test entre los modelos con brecha < 5 %.
# La regresión lineal es solo la línea base y se excluye de la selección.
pool = df_comparativo[~df_comparativo.index.str.contains("Lineal")]
candidatos = pool[pool["Brecha_R2_%"] < 5]
if candidatos.empty:
    print("⚠️ Ningún Random Forest cumple brecha < 5 %: se elige el de menor brecha.")
    mejor_nombre = pool["Brecha_R2_%"].idxmin()
else:
    mejor_nombre = candidatos["R2_Test"].idxmax()
print(f"Modelo seleccionado: {mejor_nombre}")

# Modelo final usado en el resto del notebook y en el informe
modelo_final = modelos[mejor_nombre]
res_final = df_comparativo.loc[mejor_nombre].to_dict()
res_final["Modelo"] = mejor_nombre

# Tabla lista para copiar al informe ejecutivo
print("\n--- Tabla para el informe ejecutivo ---")
print("| Métrica | Train | Test | Brecha / Variación |")
print("| :--- | :---: | :---: | :---: |")
print(f"| R² | {res_final['R2_Train']:.4f} | {res_final['R2_Test']:.4f} | {res_final['Brecha_R2_%']:.2f}% |")
print(f"| RMSE | {res_final['RMSE_Train']:.2f} € | {res_final['RMSE_Test']:.2f} € | {res_final['Var_RMSE_%']:+.2f}% |")
print(f"| MAE | {res_final['MAE_Train']:.2f} € | {res_final['MAE_Test']:.2f} € | {res_final['Var_MAE_%']:+.2f}% |")

# Contexto del error respecto al precio típico
mediana = y_test.median()
print(f"\nPrecio mediano (Test): {mediana:.2f} € | MAE Test: {res_final['MAE_Test']:.2f} € "
      f"(≈ {res_final['MAE_Test'] / mediana:.0%} del precio mediano)")

**Lectura de la comparativa**
- La regresión lineal es la línea base: limitada porque la relación entre variables y precio no es lineal.
- El Random Forest base sobreajusta de forma severa (Train muy superior a Test).
- El RF regularizado elimina casi por completo el sobreajuste.
- El RF con GridSearchCV obtiene el mejor R² y el mejor MAE en Test, con una brecha que sigue por debajo del 5 %.

## 6. Importancia de variables (*Feature Importance*)

Con *one-hot encoding*, `region` y `tipo` se reparten en decenas de columnas.
Para interpretar el modelo se **suman las importancias de las dummies de cada variable original**.

In [ ]:
importancias = pd.Series(modelo_final.feature_importances_, index=X_train.columns)


def variable_original(col):
    if col.startswith("region_"):
        return "region"
    if col.startswith("tipo_"):
        return "tipo"
    return col


imp_agrupada = importancias.groupby(variable_original).sum().sort_values(ascending=False)

display((imp_agrupada * 100).round(2).rename("Importancia (%)").to_frame())

plt.figure(figsize=(8, 4))
sns.barplot(x=imp_agrupada.values * 100, y=imp_agrupada.index, hue=imp_agrupada.index,
            palette="viridis", legend=False)
plt.title("Importancia de variables (agrupando dummies)")
plt.xlabel("Importancia (%)")
plt.ylabel("")
plt.show()

## 7. Análisis de residuos

Residuo = Real − Predicción. Un residuo **positivo** significa que el modelo **subestima** el precio.
Se analiza el comportamiento por tramos de precio para comprobar si hay sesgo en vinos de alta gama.

In [ ]:
y_pred_test = modelo_final.predict(X_test)

res = pd.DataFrame({"real": y_test.values, "pred": y_pred_test})
res["residuo"] = res["real"] - res["pred"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# a) Real vs predicho
axes[0].scatter(res["real"], res["pred"], alpha=0.4)
lim = [0, max(res["real"].max(), res["pred"].max())]
axes[0].plot(lim, lim, "r--", label="y = x")
axes[0].set_title("Valores reales vs predichos")
axes[0].set_xlabel("Precio real (€)")
axes[0].set_ylabel("Precio predicho (€)")
axes[0].legend()

# b) Residuos vs predicción
axes[1].scatter(res["pred"], res["residuo"], alpha=0.4)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_title("Residuos vs predicción")
axes[1].set_xlabel("Precio predicho (€)")
axes[1].set_ylabel("Residuo (€)")

# c) Distribución de residuos
sns.histplot(res["residuo"], bins=60, kde=True, ax=axes[2], color="gray")
axes[2].set_title("Distribución de residuos")
axes[2].set_xlabel("Residuo (€)")

plt.tight_layout()
plt.show()

In [ ]:
# Comportamiento por tramos de precio real
res["segmento"] = pd.cut(res["real"], bins=[0, 150, 500, np.inf],
                         labels=["≤ 150 €", "150 – 500 €", "> 500 €"])

tabla_segmentos = res.groupby("segmento", observed=True).agg(
    n_vinos=("residuo", "size"),
    residuo_medio=("residuo", "mean"),      # > 0 = el modelo subestima
    MAE=("residuo", lambda s: s.abs().mean()),
)
display(tabla_segmentos)

## 8. Guardado y verificación del modelo final

Se guarda el **modelo optimizado (GridSearchCV)** y la lista de columnas de entrada,
necesaria para aplicar `get_dummies` de forma consistente al predecir con datos nuevos.

In [ ]:
ruta_modelo = MODELS_DIR / "modelo_final_vinos.pkl"
ruta_columnas = MODELS_DIR / "columnas_modelo.pkl"

joblib.dump(modelo_final, ruta_modelo)
joblib.dump(list(X.columns), ruta_columnas)
print(f"Modelo guardado en: {ruta_modelo.resolve()}")

# Verificación: el modelo cargado debe dar exactamente las mismas predicciones
modelo_cargado = joblib.load(ruta_modelo)
assert np.allclose(modelo_cargado.predict(X_test), modelo_final.predict(X_test))
print("Verificación OK: el modelo cargado reproduce las predicciones.")
print(modelo_cargado)

In [ ]:
# Estructura de uno de los árboles del bosque final (limitado a 3 niveles para que sea legible)
plt.figure(figsize=(20, 10))
plot_tree(
    modelo_final.estimators_[0],
    feature_names=X_train.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)
plt.title("Estructura de un árbol del Random Forest final (profundidad 3)", fontsize=16)
plt.show()

## 9. Experimento opcional: transformación logarítmica del precio

El precio tiene cola larga y el modelo subestima los vinos caros. Se prueba entrenar sobre
`log(1 + precio)` y devolver las predicciones a euros, usando los mismos hiperparámetros óptimos.

> Es un **experimento**: no sustituye al modelo final a menos que mejore las métricas en Test **sin** aumentar el sobreajuste.

In [ ]:
rf_log = TransformedTargetRegressor(
    regressor=RandomForestRegressor(**grid_search.best_params_, random_state=RANDOM_STATE, n_jobs=-1),
    func=np.log1p,
    inverse_func=np.expm1,
)
rf_log.fit(X_train, y_train)

res_log = evaluar_modelo(rf_log, "4. RF GridSearchCV + log(precio) [experimental]")
mostrar(res_log)

display(pd.DataFrame([res_final, res_log]).set_index("Modelo"))

## 10. Conclusiones

1. **Modelo final:** Random Forest optimizado con GridSearchCV (celdas 4–5). Mejora claramente a la regresión lineal
   (R² Test ≈ 0.37) y corrige el sobreajuste severo del Random Forest base (brecha ≈ 52 %).
2. **Generalización:** la brecha de R² entre Train y Test del modelo final es inferior al 5 %, por lo que generaliza bien.
3. **Capacidad predictiva moderada:** R² Test ≈ 0.51. El modelo explica aproximadamente la mitad de la variabilidad del precio.
4. **Error en contexto:** el MAE es comparable al precio mediano del dataset, por lo que el error típico es relevante
   para vinos de gama media/baja; el RMSE está muy influido por los vinos de colección.
5. **Alta gama:** el análisis por tramos (sección 7) permite verificar el sesgo de subestimación en vinos caros.
6. **Calidad de datos:** se pierde ~16 % de las filas por nulos y la `puntuacion` casi no varía,
   lo que limita el techo de rendimiento con las variables actuales.

### Próximos pasos
- Evaluar si la transformación logarítmica (sección 9) mejora el segmento premium.
- Incorporar variables de reputación: interacción `bodega` × `region`, o *target encoding* de `bodega`.
- Agrupar categorías poco frecuentes de `region` y `tipo` en "Otras" para reducir la dimensionalidad.
- Probar *Gradient Boosting* (XGBoost / LightGBM) con regularización.
- Actualizar el Informe Ejecutivo con las tablas generadas en las secciones 5, 6 y 7.